# Information, decomposition, and robustness

In [ ]:
import itertools

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from MuxVizPy.utils import parsing
from MuxVizPy import information, decomposition, percolation, topology

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110

## Running example

Entropy reuses the office multiplex. Decomposition and node removal use separate small
networks.

In [ ]:
NODE_NAMES = ["Ada", "Ben", "Cleo", "Dan", "Eve", "Femi", "Gil", "Hana", "Ivan", "Jo"]
LAYER_NAMES = ["email", "chat", "meetings"]
N, L = len(NODE_NAMES), len(LAYER_NAMES)
idx = {name: i for i, name in enumerate(NODE_NAMES)}

INTRA = {
    "email": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ada", "Dan"), ("Ben", "Cleo"),
        ("Cleo", "Dan"), ("Dan", "Eve"), ("Eve", "Femi"), ("Femi", "Gil"),
        ("Gil", "Hana"), ("Hana", "Ivan"), ("Ivan", "Jo"), ("Jo", "Ada"),
    ],
    "chat": [
        ("Ada", "Ben"), ("Ben", "Eve"), ("Eve", "Hana"), ("Hana", "Jo"),
        ("Jo", "Cleo"), ("Cleo", "Femi"), ("Femi", "Ivan"), ("Ivan", "Dan"),
        ("Dan", "Gil"), ("Gil", "Ada"), ("Ben", "Hana"),
    ],
    "meetings": [
        ("Ada", "Ben"), ("Ada", "Cleo"), ("Ben", "Cleo"),
        ("Dan", "Eve"), ("Dan", "Femi"), ("Eve", "Femi"),
        ("Gil", "Hana"), ("Gil", "Ivan"), ("Hana", "Ivan"),
    ],
}


def undirected_rows(pairs, layer):
    """Both directions for each pair, in extended edge list form."""
    rows = []
    for a, b in pairs:
        rows.append((a, layer, b, layer, 1.0))
        rows.append((b, layer, a, layer, 1.0))
    return rows


rows = []
for layer_name, pairs in INTRA.items():
    layer = LAYER_NAMES.index(layer_name)
    rows += undirected_rows([(idx[a], idx[b]) for a, b in pairs], layer)

edges = pl.DataFrame(
    rows,
    schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
    orient="row",
)

t = parsing.build_tensor_from_dataframe(edges)
g_list = parsing.build_list_of_graphs_from_tensor(t)
node_tensor = parsing.get_node_tensor_from_network_list(g_list)

print(f"{N} nodes, {L} layers, tensor {tuple(t.shape)} with {t._nnz()} stored entries")

## Von Neumann entropy

Each layer is converted to a density matrix for von Neumann entropy.

In [ ]:
layer_adj, densities = [], []
for adj in node_tensor:
    sym = (adj + adj.T).tocsr()
    sym.data = np.ones(sym.nnz, dtype=np.float64)   # binarise, and force float
    layer_adj.append(sym)
    densities.append(parsing.build_density_bgs_from_adjacency_matrix(sym))

for name, adj, rho in zip(LAYER_NAMES, layer_adj, densities):
    laplacian = parsing.build_laplacian_matrix_from_adjacency_matrix(adj)
    trace = laplacian.diagonal().sum()
    matches = abs((rho - laplacian / trace)).max() < 1e-12
    print(f"{name:9} {adj.nnz // 2:2} edges   tr(L) = {trace:4.0f}   "
          f"rho == L / tr(L): {matches}   tr(rho) = {rho.diagonal().sum():.4f}")

In [ ]:
entropies = [information.compute_vn_entropy(rho) for rho in densities]

print(f"maximum possible entropy for {N} nodes: log({N}) = {np.log(N):.4f} nats\n")
for name, adj, h in zip(LAYER_NAMES, layer_adj, entropies):
    print(f"{name:9} {adj.nnz // 2:2} edges   entropy {h:.4f} nats")

## Jensen-Shannon divergence

`compute_js_divergence` compares two layers. Cached entropies avoid repeated work across
layer pairs.

In [ ]:
jsd = np.zeros((L, L))
for i, j in itertools.combinations(range(L), 2):
    d = information.compute_js_divergence(
        layer_adj[i], layer_adj[j], entropies[i], entropies[j]
    )
    jsd[i, j] = jsd[j, i] = d

for i, j in itertools.combinations(range(L), 2):
    print(f"{LAYER_NAMES[i]:9} vs {LAYER_NAMES[j]:9}  JSD = {jsd[i, j]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(jsd, cmap="viridis")
ax.set_xticks(range(L), LAYER_NAMES)
ax.set_yticks(range(L), LAYER_NAMES)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{jsd[i, j]:.3f}", ha="center", va="center", color="white")
ax.set_title("Jensen-Shannon divergence between layers")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

### Shortest-path comparison

In [ ]:
supra_ec = parsing.build_supra_adjacency_matrix_from_tensor(t)
sp_similarity = topology.get_SP_similarity_matrix(supra_ec, nodes=N, layers=L)

print(f"{'pair':22} {'shortest-path similarity':>26} {'JSD':>10}")
for i, j in itertools.combinations(range(L), 2):
    pair = f"{LAYER_NAMES[i]} vs {LAYER_NAMES[j]}"
    print(f"{pair:20} {sp_similarity[i, j]:>26.4f} {jsd[i, j]:>10.4f}")

## Sparse CP decomposition

`sparse_cp_decomposition` extracts node and layer patterns from the sparse
`(N, L, N, L)` tensor. `planted` has two node groups active in different layers.

In [ ]:
GROUP_A, GROUP_B = list(range(5)), list(range(5, 10))

planted_rows = (
    undirected_rows(list(itertools.combinations(GROUP_A, 2)), 0)
    + undirected_rows(list(itertools.combinations(GROUP_B, 2)), 1)
)
planted = parsing.build_tensor_from_dataframe(
    pl.DataFrame(
        planted_rows,
        schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
        orient="row",
    )
)
print(f"planted tensor {tuple(planted.shape)} with {planted._nnz()} stored entries")
print("group A (nodes 0-4) is a clique in layer 0; group B (nodes 5-9) in layer 1")

### Rank

In [ ]:
for rank in (1, 2):
    _, _, history = decomposition.sparse_cp_decomposition(
        planted, rank=rank, random_state=5, max_iter=500, backend="numpy"
    )
    err = history["reconstruction_error"]
    print(f"rank {rank}:  relative error {err[0]:.4f} -> {err[-1]:.4f}  "
          f"in {len(err)} iterations")

### Factors

The four factor matrices give source-node, source-layer, target-node, and target-layer
weights for each pattern.

In [ ]:
factors, weights, history = decomposition.sparse_cp_decomposition(
    planted, rank=2, random_state=5, max_iter=500, backend="numpy"
)
source_nodes, source_layers = factors[0], factors[1]

print("component weights:", np.round(weights, 3), "\n")
print("source-node factor, one row per component:")
print(np.round(np.abs(source_nodes), 2).T)
print("\nsource-layer factor, one row per component:")
print(np.round(np.abs(source_layers), 2).T)

### Remaining error

The residual energy is split between self-loops and all other tensor entries.

In [ ]:
dense = planted.to_dense().numpy()
approx = np.zeros_like(dense)
for r in range(len(weights)):
    approx += weights[r] * np.einsum(
        "i,j,k,l->ijkl",
        factors[0][:, r], factors[1][:, r], factors[2][:, r], factors[3][:, r],
    )

residual = dense - approx
on_diagonal = np.zeros(dense.shape, dtype=bool)
for i in range(dense.shape[0]):
    for a in range(dense.shape[1]):
        on_diagonal[i, a, i, a] = True

print(f"residual energy, total            {np.sum(residual ** 2):5.2f}")
print(f"residual energy, self-loops      {np.sum(residual[on_diagonal] ** 2):5.2f}")
print(f"residual energy, everything else {np.sum(residual[~on_diagonal] ** 2):5.2f}")

In [ ]:
for rank in (2, 5, 8, 12):
    _, _, history = decomposition.sparse_cp_decomposition(
        planted, rank=rank, random_state=5, max_iter=500, backend="numpy"
    )
    print(f"rank {rank:2}:  relative error {history['reconstruction_error'][-1]:.4f}")

### Random starts

In [ ]:
for seed in range(8):
    _, _, history = decomposition.sparse_cp_decomposition(
        planted, rank=2, random_state=seed, max_iter=500, backend="numpy"
    )
    print(f"random_state={seed}:  relative error "
          f"{history['reconstruction_error'][-1]:.4f}")

### Initialisation methods

Repeatable `hosvd` results require a NumPy seed and `random_state`.

In [ ]:
print(f"{'seed':<6}  {'init=random':>12}  {'init=hosvd':>11}")
for seed in range(8):
    errs = []
    for init in ("random", "hosvd"):
        np.random.seed(seed)          # hosvd needs this too, random_state is not enough
        _, _, history = decomposition.sparse_cp_decomposition(
            planted, rank=2, init=init, random_state=seed, max_iter=500, backend="numpy"
        )
        errs.append(history["reconstruction_error"][-1])
    print(f"{seed:<6}  {errs[0]:>12.4f}  {errs[1]:>11.4f}")

## Percolation

`get_percolation` removes nodes in a supplied order. It returns component-size curves;
`CritPoint` is where the second-largest component peaks. `bridge_graphs` has two groups
joined by one bridge.

In [ ]:
BRIDGE_N, BRIDGE_L = 12, 2
side_a, side_b = list(range(6)), list(range(6, 12))

bridge_rows = (
    undirected_rows(list(itertools.combinations(side_a, 2)), 0)
    + undirected_rows(list(itertools.combinations(side_b, 2)), 1)
    + undirected_rows([(5, 6)], 0)
)
bridge_t = parsing.build_tensor_from_dataframe(
    pl.DataFrame(
        bridge_rows,
        schema=["node.from", "layer.from", "node.to", "layer.to", "weight"],
        orient="row",
    )
)
bridge_graphs = parsing.build_list_of_graphs_from_tensor(bridge_t)
print("edges per layer:", [g.num_edges() for g in bridge_graphs])

In [ ]:
orders = {
    "bridge first": np.array([5, 6, 0, 1, 2, 3, 4, 7, 8, 9, 10, 11]),
    "bridge last": np.array([0, 1, 2, 3, 4, 7, 8, 9, 10, 11, 5, 6]),
}

results = {}
for label, order in orders.items():
    results[label] = percolation.get_percolation(
        bridge_graphs, nodes=BRIDGE_N, layers=BRIDGE_L, order=order
    )
    print(f"{label:13} largest component  {results[label]['1ComponentSize']}")
    print(f"{label:13} second largest     {results[label]['2ComponentSize']}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
removed = np.arange(BRIDGE_N) / BRIDGE_N
for label, result in results.items():
    ax.plot(removed, result["1ComponentSize"], marker="o", label=label)
ax.set_xlabel("fraction of nodes removed")
ax.set_ylabel("largest component size")
ax.set_title("Same network, two removal orders")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
for label, result in results.items():
    print(f"{label:13} critical point {result['CritPoint']:.4f}")

## Function guide

| Task | Function | Input |
|---|---|---|
| Build a density matrix | `parsing.build_density_bgs_from_adjacency_matrix` | symmetric adjacency |
| Measure layer entropy | `information.compute_vn_entropy` | density matrix |
| Compare two layers | `information.compute_js_divergence` | adjacencies and entropies |
| Split a tensor into patterns | `decomposition.sparse_cp_decomposition` | sparse tensor |
| Test node removal | `percolation.get_percolation` | graph list and removal order |